# Point Net Experimental

In this notebook I will explore the [PointNet](https://arxiv.org/abs/1612.00593) architecture in order to understand how it works. It seems for classification, the model ```PointNetCls``` and for segmentation the model ```PointNetDenseCls``` is used. There are other variants of models (```PointNetfeat```, ```STNkd```, ```STN3d```). As it is the simpler model, we will start with the classification model.

## PointNetCls

In [1]:
import torch

In [2]:
batchSize = 32
num_points = 2500
workers = 4
nepoch = 250
outf  = 'cls' # output folder
model = ''
dataset = '' # dataset path
dataset_type = 'shapenet'
feature_transform = True

In [3]:
blue = lambda x: '\033[94m' + x + '\033[0m'
print(blue("Lol das ist ja in blau!")) 

Lol das ist ja in blau!


In [4]:
manualSeed = 42
torch.manual_seed(manualSeed)

After setting preliminary stuff, first a dataset has to be created using the ```ShapeNetDataset``` class. It will be implemented here

START WITH NEW GITHUB REPO Pointnet_Pointnet2_pytorch which contains both pointnet models but in pytorch!!
Especially look out for the final layers and the how to change the loss to regression.


# PointNet from Pointnet_Pointnet2

Starting with the new repo, we will go through PointNet in order to understand how the latent representation z can be reconstructed. Therefore we will scrutinize the ```train_classification.py``` file using the ```pointnet_cls.py``` model.

The classification script works as follows:
- Check Cuda devices
- Create experiment logging directory
- Create a logger for detailed logging info
- Load the data -> data output format is: point_set, target_class_label -> **We have to adapt the dataloader** so output is: point_set, latent_vector
- Then the model and loss are loaded, also the python files necessary for training are copied into the experiment directory -> **Here we have to change the loss to MSE**
- Then the script checks for checkpoints and initializes Adam optimizer and learning rate scheduler

#### Training
- Before being processed, the points are randomly droped out, scaled and shifted (Why?)
- Then they are processed by the Classifier
- The ```PointNetEncoder``` produces the global feature vector and contains the important model architecture -> **One can take this encoder and use it for DeepCAD**
- First the encoder predicts a matrix for affine transformation of the point features (T-Net) -> This transformation matrix is part of the loss calculation, where the loss tries to keep the transformation matrices as orthogonal as possible ($A \cdot A^T = I$)
- Then a per point MLP/1D Convolution per point (same as MLP) is applied to increase feature space and another transformation matrix is predicted -> Actually only this one goes into the loss, not the point features
- Finally you get the global feature vector

#### Next steps
- Understand pointnet++ (or try first with simpler pointnet?)
- Change loss to MSE regression loss
- Change data to: Input->PC, Target -> Latent vector
- Find a good workflow, i.e. Should I just copy the important code parts from the github repo or should I fork and change some things? What are best practices?

# PointNet++ from Pointnet_Pointnet2

After understanding how PointNet works, let's scrutinize its successor PointNet++. In contrast to its predecessor, PointNet++ uses a hierarchical structure to obtain the global feature vector, thereby putting more emphasis on local features. The setup script has the same structure as in PointNet which is why we will start by looking at the training.

### Training
- The script first checks if the points contain normals or not. In our case we will have no normals. The points are sent to the first _set abstraction_ layer

#### Set abstraction layer 1
- Then centroid points need to be sampled using iterative farthest point sampling (FPS) -> This algorithm basically always chooses the point which is the furthest away from a set of points and results in a good coverage of the point cloud with centroid points -> Result: For each batch sample you get a specific number of centroid points with their index
- Then the script retrieves the coordinates of the points behind the sampled centroid indices
- Next, the indices of the points within a specified radius of each centroid points are found and the coordinates are retrieved
- Thereby you have the following final shape of the data: [B, npoint, nsample, C]
    - _B_: Batch size
    - _npoint_: Number of centroid points
    - _nsample_: Number of points within a certain radius to each centroid point
    - _C_: Coordinates
- The script then calculates the norm between the grouped points and their respective centroid point
- Finally the centroid points and the norm of the grouped points to the centroid points are returned

#### 1D Convolution/Fully Connected Layers
- 1D convolutions (three with output size: 64,64,128) with a kernel of size 1x1 are applied on the norm of the grouped points
    - this is equivalent to applying a fully connected layer on the 3 coordinates of each point of the norm
    - the output has then instead of 3 channels 128 channels
- Then the symmetric function (max) is applied so that we can find the most meaningful features from the _nsample_ points which are within a radius to the centroid points -> **The convolutions and the max functions basically resemble the original PointNet**
- Returned are the centroid points with their coordinates and the features from the norm (now 128 channels)

### Next set abstraction layer
- in the next sa layer, the obtained centroids and the norm with enhanced channels are passed
- Again FPS collects centroid points and ball query finds points within a radius of that and finally the norm is calculated
- The **difference** is now, that we have the extracted (maximum) features of each centroid point of the stage before
- The script now concatenates the new calculated norm (3 coordinates) with the enhanced features of the points lying within a radius to the centroid points (128 features) (those points were centroid points in the previous stage)
- In the final set abstraction layer, no further centroid points are sampled and no grouping takes place
- Only the final centroid points are concatenated with the point embeddings, which are returned and processed by the convolutional layers and the max function, resulting in a 1024 dimensional vector for each point cloud (The point dimension is collapsed due to the max function)

### Final linear layers
- The 1024 dimensional vector per batch sample is processed by 3 fully connected layers to get the final prediction

### Farthest Point sampling
The algorithm is quite simple. You start with a point cloud comprising N points and iteratively select a point until you have up to S samples. You have two sets which we will denote sampled and remaining and you choose a point as follows:
- For each point in remaining find its nearest neighbour in sampled, saving the distance.
- Select the point in remaining whose nearest neighbour distance is the largest and move it from remaining to sampled.


In [19]:
def farthest_point_sample(xyz, npoint):
    """
    Input:
        xyz: pointcloud data, [B, N, 3]
        npoint: number of samples
    Return:
        centroids: sampled pointcloud index, [B, npoint]
    """
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids

def index_points(points, idx):
    """

    Input:
        points: input points data, [B, N, C]
        idx: sample index data, [B, S]
    Return:
        new_points:, indexed points data, [B, S, C]
    """
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

def square_distance(src, dst):
    """
    Calculate Euclid distance between each two points.

    src^T * dst = xn * xm + yn * ym + zn * zm；
    sum(src^2, dim=-1) = xn*xn + yn*yn + zn*zn;
    sum(dst^2, dim=-1) = xm*xm + ym*ym + zm*zm;
    dist = (xn - xm)^2 + (yn - ym)^2 + (zn - zm)^2
         = sum(src**2,dim=-1)+sum(dst**2,dim=-1)-2*src^T*dst

    Input:
        src: source points, [B, N, C]
        dst: target points, [B, M, C]
    Output:
        dist: per-point square distance, [B, N, M]
    """
    B, N, _ = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, -1).view(B, N, 1)
    dist += torch.sum(dst ** 2, -1).view(B, 1, M)
    return dist

def query_ball_point(radius, nsample, xyz, new_xyz):
    """
    Input:
        radius: local region radius
        nsample: max sample number in local region
        xyz: all points, [B, N, 3]
        new_xyz: query points, [B, S, 3]
    Return:
        group_idx: grouped points index, [B, S, nsample]
    """
    device = xyz.device
    B, N, C = xyz.shape
    _, S, _ = new_xyz.shape
    group_idx = torch.arange(N, dtype=torch.long).to(device).view(1, 1, N).repeat([B, S, 1])
    sqrdists = square_distance(new_xyz, xyz)
    group_idx[sqrdists > radius ** 2] = N
    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]
    group_first = group_idx[:, :, 0].view(B, S, 1).repeat([1, 1, nsample])
    mask = group_idx == N
    group_idx[mask] = group_first[mask]
    return group_idx

In [75]:
B = 1
S = 20
C = 3

xyz = torch.rand(B,100,C)
print(f"xyz\t\t\tPoint cloud with 100 points (bs={B}): {xyz.shape}")

fps_idx = farthest_point_sample(xyz, S)
print(f"fps_idx\t\t\tIndices of {S} centroid points: {fps_idx.shape}")

new_xyz = index_points(xyz, fps_idx)
print(f"new_xyz\t\t\tCoordinates of the {S} centroid points: {new_xyz.shape}")

idx = query_ball_point(0.2, 32, xyz, new_xyz)
print(f"idx\t\t\tIndices of 32 points within a 0.2 radius to the centroid points: {idx.shape}")

grouped_xyz = index_points(xyz, idx)
print(f"grouped_xyz\t\tCoordinates of the 32 points within a 0.2 radius to the centroid points: {grouped_xyz.shape}")

grouped_xyz_norm = grouped_xyz - new_xyz.view(B, S, 1, C)
print(f"grouped_xyz_norm\tSubstract the coordinates of the centroid points from the grouped points to get the norm: {grouped_xyz_norm.shape}")

xyz			Point cloud with 100 points (bs=1): torch.Size([1, 100, 3])
fps_idx			Indices of 20 centroid points: torch.Size([1, 20])
new_xyz			Coordinates of the 20 centroid points: torch.Size([1, 20, 3])
idx			Indices of 32 points within a 0.2 radius to the centroid points: torch.Size([1, 20, 32])
grouped_xyz		Coordinates of the 32 points within a 0.2 radius to the centroid points: torch.Size([1, 20, 32, 3])
grouped_xyz_norm	Substract the coordinates of the centroid points from the grouped points to get the norm: torch.Size([1, 20, 32, 3])


In [76]:
new_xyz  = new_xyz
new_points = grouped_xyz_norm
new_points.shape

torch.Size([1, 20, 32, 3])

In [77]:
new_points = new_points.permute(0, 3, 2, 1)
new_points.shape

torch.Size([1, 3, 32, 20])

In [78]:
import torch.nn as nn
import torch.nn.functional as F

mlp = [64, 64, 128]
in_channel = 3

mlp_convs = nn.ModuleList()
mlp_bns = nn.ModuleList()
last_channel = in_channel
for out_channel in mlp:
    mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
    mlp_bns.append(nn.BatchNorm2d(out_channel))
    last_channel = out_channel

In [79]:
for i, conv in enumerate(mlp_convs):
    print(i, conv)
    bn = mlp_bns[i]
    new_points =  F.relu(bn(conv(new_points)))
new_points.shape

0 Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
1 Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
2 Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))


torch.Size([1, 128, 32, 20])

In [80]:
new_points = torch.max(new_points, 2)[0]
new_points.shape

torch.Size([1, 128, 20])

In [81]:
new_xyz = new_xyz.permute(0, 2, 1)
new_xyz.shape

torch.Size([1, 3, 20])

In [82]:
xyz = new_xyz.permute(0, 2, 1)
points = new_points.permute(0, 2, 1)


fps_idx = farthest_point_sample(xyz, S) # [B, npoint, C]
new_xyz = index_points(xyz, fps_idx)
idx = query_ball_point(0.4, 10, xyz, new_xyz)
grouped_xyz = index_points(xyz, idx) # [B, npoint, nsample, C]
grouped_xyz_norm = grouped_xyz - new_xyz.view(B, S, 1, C)

In [83]:
new_xyz.shape

torch.Size([1, 20, 3])

In [84]:
grouped_points = index_points(points, idx)

In [85]:
grouped_points.shape

torch.Size([1, 20, 10, 128])

In [86]:
grouped_xyz_norm.shape

torch.Size([1, 20, 10, 3])